# Clinical Change-from-Baseline Table with `row_pivot()`

This notebook is a worked example of a classic clinical descriptive-statistics
table:

> For each **Visit**, by **Treatment Arm**, show `n`, `Mean (SD)`,
> `Median (Q1, Q3)`, and `Min, Max` — for both the observed **Value** and the
> **Change from Baseline** — as separate rows under each visit, with Arm
> pivoted into columns.

It demonstrates `row_pivot`, the `simple_table()`/`gt_table()` argument added
in [#62](https://github.com/BFalquet/pyMyriad/issues/62) for stacking a
*subset* of statistics as rows instead of columns — the piece
`pivot_statistics=True` alone doesn't provide.


In [1]:
import numpy as np
import pandas as pd
from great_tables import GT

from pyMyriad import AnalysisTree, simple_table

## 1. Simulate CDISC-like lab data

One row per subject per visit. `AVISIT` and `ARM` are both ordered
categoricals, so the table below comes out in chronological/arm order rather
than alphabetical (see [#61](https://github.com/BFalquet/pyMyriad/issues/61)).

In [2]:
rng = np.random.default_rng(42)

ARMS = ["Placebo", "Active 10 mg"]
VISITS = ["Baseline", "Week 4", "Week 8", "Week 12"]
N_SUBJECTS = 40

subjects = pd.DataFrame(
    {
        "USUBJID": [f"S{i:03d}" for i in range(N_SUBJECTS)],
        "ARM": rng.choice(ARMS, size=N_SUBJECTS),
    }
)

rows = []
for subj in subjects.itertuples():
    baseline = rng.normal(45, 8)
    drift = -6 if subj.ARM == "Active 10 mg" else -1  # active arm trends down
    for visit_idx, visit in enumerate(VISITS):
        aval = baseline + drift * visit_idx / 3 + rng.normal(0, 3)
        rows.append(
            {"USUBJID": subj.USUBJID, "ARM": subj.ARM, "AVISIT": visit, "AVAL": aval}
        )

df = pd.DataFrame(rows)
df["AVISIT"] = pd.Categorical(df["AVISIT"], categories=VISITS, ordered=True)
df["ARM"] = pd.Categorical(df["ARM"], categories=ARMS, ordered=True)
df.head()

,USUBJID,ARM,AVISIT,AVAL
0,S000,Placebo,Baseline,41.478312
1,S000,Placebo,Week 4,46.855392
2,S000,Placebo,Week 8,42.390846
3,S000,Placebo,Week 12,41.236118
4,S001,Active 10 mg,Baseline,43.779859


## 2. The problem: `pivot_statistics=True` puts *every* statistic in its own column

Build a tree computing the raw statistics we need (`n`, `mean`, `sd`,
`median`, `q1`, `q3`, `min`, `max`), pivot `Arm` to columns, and turn on
`pivot_statistics`. Every statistic becomes its own column — 8 statistics x
2 arms = 16 data columns, all on a single row per visit.

In [3]:
value_tree = (
    AnalysisTree()
    .split_by("df.AVISIT", label="Visit")
    .split_by("df.ARM", label="Arm")
    .analyze_by(
        n=lambda df: len(df),
        mean=lambda df: round(np.mean(df.AVAL), 1),
        sd=lambda df: round(np.std(df.AVAL, ddof=1), 1),
        median=lambda df: round(np.median(df.AVAL), 1),
        q1=lambda df: round(np.percentile(df.AVAL, 25), 1),
        q3=lambda df: round(np.percentile(df.AVAL, 75), 1),
        min=lambda df: round(np.min(df.AVAL), 1),
        max=lambda df: round(np.max(df.AVAL), 1),
    )
)
value_result = value_tree.run(df)

wide = simple_table(value_result, by="Arm", pivot_statistics=True)
print(f"{len(wide.columns)} columns: {list(wide.columns)}")
wide

18 columns: ['_Level_0', '_Level_1', 'Placebo||max', 'Placebo||mean', 'Placebo||median', 'Placebo||min', 'Placebo||n', 'Placebo||q1', 'Placebo||q3', 'Placebo||sd', 'Active 10 mg||max', 'Active 10 mg||mean', 'Active 10 mg||median', 'Active 10 mg||min', 'Active 10 mg||n', 'Active 10 mg||q1', 'Active 10 mg||q3', 'Active 10 mg||sd']


,_Level_0,_Level_1,Placebo||max,Placebo||mean,Placebo||median,Placebo||min,Placebo||n,Placebo||q1,Placebo||q3,Placebo||sd,Active 10 mg||max,Active 10 mg||mean,Active 10 mg||median,Active 10 mg||min,Active 10 mg||n,Active 10 mg||q1,Active 10 mg||q3,Active 10 mg||sd
0,Visit,Baseline,56.1,45.1,41.5,30.5,17,40.6,51.1,7.0,60.9,46.2,46.7,29.5,23,39.5,51.0,8.8
1,,Week 4,53.9,43.8,43.5,31.7,17,38.6,48.6,6.2,60.1,44.3,44.7,28.4,23,39.1,48.0,8.2
2,,Week 8,58.1,44.5,45.2,32.6,17,40.5,50.2,6.8,59.2,42.9,43.5,30.6,23,38.4,47.4,7.7
3,,Week 12,55.7,43.1,42.6,32.7,17,36.7,49.8,7.2,58.0,40.2,39.3,24.7,23,35.8,43.9,7.9


## 3. `row_pivot`: combine a chosen subset of statistics into labelled rows

`row_pivot` maps a new row label to either:

- **a list of statistic names** — a single name is used as-is, multiple names
  are joined with `", "` (e.g. `["min", "max"]` → `"30.1, 60.2"`)
- **a callable**, dispatched by parameter name exactly like `analyze_by()` —
  its parameters name the statistics to fetch, and its return value is used
  directly, giving full control over formatting (e.g.
  `lambda mean, sd: f"{mean} ({sd})"` → `"45.1 (7.0)"`)

Statistics not referenced by any entry are dropped.

In [4]:
ROW_PIVOT = {
    "n": ["n"],
    "Mean (SD)": lambda mean, sd: f"{mean} ({sd})",
    "Median (Q1, Q3)": lambda median, q1, q3: f"{median} ({q1}, {q3})",
    "Min, Max": lambda min, max: f"{min}, {max}",
}

simple_table(value_result, by="Arm", pivot_statistics=True, row_pivot=ROW_PIVOT)

,_Level_0,_Level_1,Statistic,Placebo,Active 10 mg
0,Visit,Baseline,n,17,23
1,,,Mean (SD),45.1 (7.0),46.2 (8.8)
2,,,"Median (Q1, Q3)","41.5 (40.6, 51.1)","46.7 (39.5, 51.0)"
3,,,"Min, Max","30.5, 56.1","29.5, 60.9"
4,,Week 4,n,17,23
5,,,Mean (SD),43.8 (6.2),44.3 (8.2)
6,,,"Median (Q1, Q3)","43.5 (38.6, 48.6)","44.7 (39.1, 48.0)"
7,,,"Min, Max","31.7, 53.9","28.4, 60.1"
8,,Week 8,n,17,23
9,,,Mean (SD),44.5 (6.8),42.9 (7.7)


## 4. Adding "Change from Baseline" as a second pivoted dimension

`row_pivot` only pivots **one** column dimension (`by=`). "Value" vs "Change
from Baseline" isn't a tree split though — both are computed from the *same*
rows, just with a different metric.

The trick: reshape the data into long form with a literal `MEASURE` column
holding whichever metric applies. That turns "Value vs Change" into a real,
poolable split, which can then be pivoted *together* with `Arm` via
`by=["Arm", "Measure"]` — one tree, one `analyze_by()` call, one
`simple_table(row_pivot=...)` call for the whole table.

In [5]:
baseline_lookup = df.loc[df.AVISIT == "Baseline"].set_index("USUBJID")["AVAL"]
df["CHANGE"] = df.AVAL - df.USUBJID.map(baseline_lookup)

long_df = pd.concat(
    [
        df.assign(METRIC=df.AVAL, MEASURE="Value"),
        df.assign(METRIC=df.CHANGE, MEASURE="Change"),
    ],
    ignore_index=True,
)
long_df["MEASURE"] = pd.Categorical(
    long_df["MEASURE"], categories=["Value", "Change"], ordered=True
)
long_df.head()

,USUBJID,ARM,AVISIT,AVAL,CHANGE,METRIC,MEASURE
0,S000,Placebo,Baseline,41.478312,0.000000,41.478312,Value
1,S000,Placebo,Week 4,46.855392,5.377079,46.855392,Value
2,S000,Placebo,Week 8,42.390846,0.912534,42.390846,Value
3,S000,Placebo,Week 12,41.236118,-0.242195,41.236118,Value
4,S001,Active 10 mg,Baseline,43.779859,0.000000,43.779859,Value


In [6]:
tree = (
    AnalysisTree()
    .split_by("df.AVISIT", label="Visit")
    .split_by("df.ARM", label="Arm")
    .split_by("df.MEASURE", label="Measure")
    .analyze_by(
        n=lambda df: len(df),
        mean=lambda df: round(np.mean(df.METRIC), 1),
        sd=lambda df: round(np.std(df.METRIC, ddof=1), 1),
        median=lambda df: round(np.median(df.METRIC), 1),
        q1=lambda df: round(np.percentile(df.METRIC, 25), 1),
        q3=lambda df: round(np.percentile(df.METRIC, 75), 1),
        min=lambda df: round(np.min(df.METRIC), 1),
        max=lambda df: round(np.max(df.METRIC), 1),
    )
)
result = tree.run(long_df)

table = simple_table(
    result, by=["Arm", "Measure"], pivot_statistics=True, row_pivot=ROW_PIVOT
)
table = table.drop(columns=["_Level_0"]).rename(columns={"_Level_1": "Visit"})
table = table.rename(
    columns={c: c.replace(" > ", "||") for c in table.columns if " > " in c}
)
table = table[
    ["Visit", "Statistic"]
    + [f"{arm}||{measure}" for arm in ARMS for measure in ("Value", "Change")]
]
table

,Visit,Statistic,Placebo||Value,Placebo||Change,Active 10 mg||Value,Active 10 mg||Change
0,Baseline,n,17,17,23,23
1,,Mean (SD),45.1 (7.0),0.0 (0.0),46.2 (8.8),0.0 (0.0)
2,,"Median (Q1, Q3)","41.5 (40.6, 51.1)","0.0 (0.0, 0.0)","46.7 (39.5, 51.0)","0.0 (0.0, 0.0)"
3,,"Min, Max","30.5, 56.1","0.0, 0.0","29.5, 60.9","0.0, 0.0"
4,Week 4,n,17,17,23,23
5,,Mean (SD),43.8 (6.2),-1.4 (3.9),44.3 (8.2),-1.9 (3.1)
6,,"Median (Q1, Q3)","43.5 (38.6, 48.6)","-2.2 (-3.4, 1.2)","44.7 (39.1, 48.0)","-1.9 (-3.5, -0.5)"
7,,"Min, Max","31.7, 53.9","-8.3, 5.7","28.4, 60.1","-7.0, 6.6"
8,Week 8,n,17,17,23,23
9,,Mean (SD),44.5 (6.8),-0.7 (3.7),42.9 (7.7),-3.2 (3.2)


## 5. Great Tables version

Add Arm spanners over the Value/Change column pairs, and suppress the
repeated Visit label for the 3 extra rows under each visit.

In [7]:
display_table = table.copy()
display_table.loc[display_table["Visit"].duplicated(), "Visit"] = ""

arm_columns = {
    arm: [c for c in display_table.columns if c.startswith(f"{arm}||")] for arm in ARMS
}

gt = GT(display_table).tab_header(
    title="ALT (U/L) by Visit and Treatment Arm",
    subtitle="n; Mean (SD); Median (Q1, Q3); Min, Max - Value and Change from Baseline",
)
for arm, cols in arm_columns.items():
    gt = gt.tab_spanner(label=arm, columns=cols)
gt = gt.cols_label(**{c: c.split("||", 1)[1] for cols in arm_columns.values() for c in cols})
gt = gt.cols_align(align="center", columns=[c for cols in arm_columns.values() for c in cols])
gt

GT(_tbl_data=       Visit        Statistic     Placebo||Value    Placebo||Change  \
0   Baseline                n                 17                 17   
1                   Mean (SD)         45.1 (7.0)          0.0 (0.0)   
2             Median (Q1, Q3)  41.5 (40.6, 51.1)     0.0 (0.0, 0.0)   
3                    Min, Max         30.5, 56.1           0.0, 0.0   
4     Week 4                n                 17                 17   
5                   Mean (SD)         43.8 (6.2)         -1.4 (3.9)   
6             Median (Q1, Q3)  43.5 (38.6, 48.6)   -2.2 (-3.4, 1.2)   
7                    Min, Max         31.7, 53.9          -8.3, 5.7   
8     Week 8                n                 17                 17   
9                   Mean (SD)         44.5 (6.8)         -0.7 (3.7)   
10            Median (Q1, Q3)  45.2 (40.5, 50.2)    0.9 (-3.5, 2.1)   
11                   Min, Max         32.6, 58.1          -7.0, 4.7   
12   Week 12                n                 17                 17   
13                  Mean (SD)         43.1 (7.2)         -2.0 (3.0)   
14            Median (Q1, Q3)  42.6 (36.7, 49.8)  -2.3 (-3.9, -0.2)   
15                   Min, Max         32.7, 55.7          -8.2, 2.1   

   Active 10 mg||Value Active 10 mg||Change  
0                   23                   23  
1           46.2 (8.8)            0.0 (0.0)  
2    46.7 (39.5, 51.0)       0.0 (0.0, 0.0)  
3           29.5, 60.9             0.0, 0.0  
4                   23                   23  
5           44.3 (8.2)           -1.9 (3.1)  
6    44.7 (39.1, 48.0)    -1.9 (-3.5, -0.5)  
7           28.4, 60.1            -7.0, 6.6  
8                   23                   23  
9           42.9 (7.7)           -3.2 (3.2)  
10   43.5 (38.4, 47.4)    -4.1 (-5.7, -0.3)  
11          30.6, 59.2            -9.1, 1.2  
12                  23                   23  
13          40.2 (7.9)           -6.0 (4.2)  
14   39.3 (35.8, 43.9)    -5.5 (-8.6, -3.1)  
15          24.7, 58.0           -16.4, 1.8  , _body=<great_tables._gt_data.Body object at 0x10f27ebd0>, _boxhead=Boxhead([ColInfo(var='Visit', type=<ColInfoTypeEnum.default: 1>, column_label='Visit', column_align='left', column_width=None), ColInfo(var='Statistic', type=<ColInfoTypeEnum.default: 1>, column_label='Statistic', column_align='left', column_width=None), ColInfo(var='Placebo||Value', type=<ColInfoTypeEnum.default: 1>, column_label='Value', column_align='center', column_width=None), ColInfo(var='Placebo||Change', type=<ColInfoTypeEnum.default: 1>, column_label='Change', column_align='center', column_width=None), ColInfo(var='Active 10 mg||Value', type=<ColInfoTypeEnum.default: 1>, column_label='Value', column_align='center', column_width=None), ColInfo(var='Active 10 mg||Change', type=<ColInfoTypeEnum.default: 1>, column_label='Change', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x112532720>, _spanners=Spanners([SpannerInfo(spanner_id='Placebo', spanner_level=0, spanner_label='Placebo', spanner_units=None, spanner_pattern=None, vars=['Placebo||Value', 'Placebo||Change'], built=None), SpannerInfo(spanner_id='Active 10 mg', spanner_level=0, spanner_label='Active 10 mg', spanner_units=None, spanner_pattern=None, vars=['Active 10 mg||Value', 'Active 10 mg||Change'], built=None)]), _heading=Heading(title='ALT (U/L) by Visit and Treatment Arm', subtitle='n; Mean (SD); Median (Q1, Q3); Min, Max - Value and Change from Baseline', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x11258bb60>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x111c808c0>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x112137bc0>, _formats=[], _substitutions=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, cat

## Notes

- Before `row_pivot`, this table required computing every statistic as its
  own pre-formatted *string* (e.g. `"45.1 (7.0)"` built inside the
  `analyze_by()` lambda itself), then hand-rolling a `pd.concat()` of one
  block per statistic group to reshape the wide table into the row-per-
  statistic layout shown above (see `examples/lab_change_from_baseline_table_v3.py`,
  step 4, for that version). `row_pivot` lets `analyze_by()` stay focused on
  *raw* numbers, and does the combining/labelling/ordering/reshaping in one
  call.
- `row_pivot` requires `pivot_statistics=True`, and is not currently
  supported with `cascade_table()` / `gt_table(cascade=True)`.
- See `format_statistics()` (in `pyMyriad.tabular`) as a lighter-weight
  alternative when you only need to combine statistics into formatted values
  without reordering or selecting a subset — it mutates the tree directly
  with `str.format()` specs, before any table is built.